In [ ]:
import os
import csv
import glob
import json
import math
import urllib.request, urllib.error
from datetime import datetime
from collections import defaultdict, namedtuple

# ======== Config ========
BASE_URL   = "https://litellm.sph-prod.ethz.ch/"
EMBED_URL  = BASE_URL + "v1/embeddings"
EMBED_MODEL = "text-embedding-3-large"   # or "text-embedding-3-large"
API_KEY    = "sk-BZZUHD8h72R2zchGXH7e5A"  # prefer env var
HEADERS    = {"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"}

# Which CSVs to read (adjust as needed)
INPUT_GLOB = "*.csv"   # e.g., "runs/*.csv"

# Output
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
OUTPUT_CSV = f"similarities_same_prompt_{timestamp}.csv"

# Columns we expect
REQUIRED_COLS = {
    "Prompt Number", "Prompt Text", "Model Response"
}

Row = namedtuple("Row", "file prompt_number response prompt_text meta")

# ======== Helpers ========
def l2_normalize(vec):
    norm = math.sqrt(sum(x*x for x in vec)) or 1.0
    return [x / norm for x in vec]

def embed_batch(texts):
    payload = {"model": EMBED_MODEL, "input": texts}
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(EMBED_URL, data=data, headers=HEADERS, method="POST")
    with urllib.request.urlopen(req, timeout=60) as resp:
        out = json.loads(resp.read().decode("utf-8"))
    embs = [l2_normalize(item["embedding"]) for item in out["data"]]
    return embs

def cosine(u, v):  # vectors already L2-normalized
    return sum(a*b for a,b in zip(u,v))

# ======== 1) Discover input files ========
print("CWD:", os.getcwd())
all_files = ['../../data/raw/responses/gpt_responses.csv','../../data/raw/responses/claude_responses.csv']
# Optional: skip previously generated similarity files
if not all_files:
    raise SystemExit("No input CSVs found. Adjust INPUT_GLOB.")

print("Found CSV files:", all_files)

# ======== 2) Load rows grouped by Prompt Number ========
by_prompt = defaultdict(list)  # prompt_number -> [Row,...]

for path in all_files:
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        missing = REQUIRED_COLS - set(reader.fieldnames or [])
        if missing:
            print(f"⚠️ Skipping {path} (missing columns: {missing})")
            continue
        for r in reader:
            pn = str(r.get("Prompt Number", "")).strip()
            text = (r.get("Model Response") or "").strip()
            if not pn or not text:
                continue
            row = Row(
                file=os.path.basename(path),
                prompt_number=pn,
                response=text,
                prompt_text=(r.get("Prompt Text") or "").strip(),
                meta={k: r.get(k) for k in [
                    "age","ethnicity","gender","education","diagnosis","treatment_outlook","Generation Time"
                ]}
            )
            by_prompt[pn].append(row)

total_prompts = len(by_prompt)
print(f"Loaded prompts: {total_prompts}")

# ======== 3) Embed all unique responses (cache to avoid repeats) ========
text_to_vec = {}
BATCH = 64

def get_vec(text):
    return text_to_vec[text]

# collect unique texts
unique_texts = list({row.response for rows in by_prompt.values() for row in rows})

# embed in batches
for i in range(0, len(unique_texts), BATCH):
    chunk = unique_texts[i:i+BATCH]
    embs = embed_batch(chunk)
    for t, e in zip(chunk, embs):
        text_to_vec[t] = e
print(f"Embedded {len(text_to_vec)} unique responses.")

# ======== 4) Compute pairwise similarities for same Prompt Number ========
out_rows = []
for pn, rows in by_prompt.items():
    # compare across files only (i < j)
    n = len(rows)
    if n < 2:
        continue

    # Optional sanity: check that Prompt Text is the same across files
    # (skip if you know they are identical)
    # pt_set = {r.prompt_text for r in rows}
    # if len(pt_set) > 1:
    #     print(f"Note: Prompt Text differs for Prompt Number {pn}")

    for i in range(n):
        for j in range(i+1, n):
            rA, rB = rows[i], rows[j]
            vA, vB = get_vec(rA.response), get_vec(rB.response)
            sim = cosine(vA, vB)
            out_rows.append({
                "Prompt Number": pn,
                "File A": rA.file,
                "File B": rB.file,
                "Cosine Similarity": f"{sim:.6f}",
                # Optional diagnostics (uncomment if useful):
                # "Len A": len(rA.response),
                # "Len B": len(rB.response),
                # "Generation Time A": rA.meta.get("Generation Time"),
                # "Generation Time B": rB.meta.get("Generation Time"),
            })

print(f"Computed {len(out_rows)} pairwise similarities.")

# ======== 5) Save CSV ========
fieldnames = ["Prompt Number","File A","File B","Cosine Similarity"]
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(out_rows)

print(f"✅ Saved: {OUTPUT_CSV}")

#Rough guide: >0.85 ≈ very similar; 0.70–0.85 ≈ moderately similar; <0.70 ≈ different.


CWD: c:\Users\User\Desktop\Desktop\Uni\Semester7\Ethics\AI4Good\Code\Response_Similarity
Found CSV files: ['../../Prompts_And_Response/initial_prompts_with_responses_gpt.csv', '../../Prompts_And_Response/initial_prompts_with_responses_claude.csv']
Loaded prompts: 156
Embedded 312 unique responses.
Computed 156 pairwise similarities.
✅ Saved: similarities_same_prompt_2025-08-27_16-57-39.csv
